In [1]:
print(121)

121


In [2]:
import pandas as pd
import os
import requests
from rdkit import Chem
from rdkit.Chem import AllChem

# Create directories for clean file organization
os.makedirs("docking_inputs/ligands", exist_ok=True)
os.makedirs("docking_inputs/protein", exist_ok=True)


In [4]:

# =====================================================================
# STEP 1: DOWNLOAD ERBB2 (HER2) 3D CRYSTAL STRUCTURE
# =====================================================================
# Using PDB ID: 3RCD (Crystal structure of the ErbB2 kinase domain)
pdb_id = "3RCD"
protein_url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

print(f"--- STEP 1: DOWNLOADING ERBB2 PROTEIN STRUCTURE ({pdb_id}) ---")
response = requests.get(protein_url)
if response.status_code == 200:
    protein_path = f"docking_inputs/protein/{pdb_id}_raw.pdb"
    with open(protein_path, "w") as f:
        f.write(response.text)
    print(f"Successfully saved raw protein structure to {protein_path}")
else:
    print("❌ Error: Failed to download PDB structure.")

# =====================================================================
# STEP 2: CONVERT 2D LEAD SMILES TO 3D CONFORMATIONS (.PDB)
# =====================================================================
print("\n--- STEP 2: GENERATING 3D LIGAND CONFORMATIONS ---")
# Load your drug-like hits saved from the previous task
df_leads = pd.read_csv("data/final_drug_like_her2_hits.csv")

print(f"Processing {len(df_leads)} candidate molecules for 3D generation...")

for idx, row in df_leads.iterrows():
    cid = row['CID']
    smiles = row['ConnectivitySMILES']
    
    try:
        # Parse SMILES and add explicit hydrogens
        mol = Chem.MolFromSmiles(smiles)
        mol_with_h = Chem.AddHs(mol)
        
        # Embed 3D coordinates using ETKDG method
        embed_status = AllChem.EmbedMolecule(mol_with_h, AllChem.ETKDGv3())
        
        if embed_status == 0:
            # Optimize structural geometry using MMFF94 force field
            AllChem.MMFFOptimizeMolecule(mol_with_h, maxIters=500)
            
            # Save out as PDB format
            ligand_pdb_path = f"docking_inputs/ligands/CID_{cid}.pdb"
            Chem.MolToPDBFile(mol_with_h, ligand_pdb_path)
        else:
            print(f"  ⚠️ Warning: 3D embedding failed for CID {cid}")
    except Exception as e:
        print(f"  ❌ Error processing CID {cid}: {e}")

print(f"3D coordinates generated for valid structures inside 'docking_inputs/ligands/'")

# =====================================================================
# STEP 3: CREATE AUTODOCK VINA CONFIGURATION FILE
# =====================================================================
print("\n--- STEP 3: CREATING AUTODOCK VINA CONFIGURATION ---")

# Grid box parameters targeting the catalytic ATP-binding pocket of ERBB2 (3RCD)
config_content = """# AutoDock Vina Grid Box Configuration File
# Target: ERBB2 / HER2 Kinase Domain (PDB: 3RCD)

receptor = protein/3RCD_prepared.pdbqt

# Active site coordinates (Centered on the hinge region/binding pocket)
center_x = 13.52
center_y = 28.14
center_z = 9.68

# Search space dimensions (Angstroms)
size_x = 24.0
size_y = 24.0
size_z = 24.0

# Search exhaustiveness parameters
exhaustiveness = 8
num_modes = 9
energy_range = 3.0
"""

config_path = "docking_inputs/vina_config.txt"
with open(config_path, "w") as f:
    f.write(config_content)

print(f"AutoDock Vina configuration file compiled and saved to '{config_path}'")
print("\n🎉 Task complete! Ready for local execution setup.")

--- STEP 1: DOWNLOADING ERBB2 PROTEIN STRUCTURE (3RCD) ---
Successfully saved raw protein structure to docking_inputs/protein/3RCD_raw.pdb

--- STEP 2: GENERATING 3D LIGAND CONFORMATIONS ---
Processing 2312 candidate molecules for 3D generation...
3D coordinates generated for valid structures inside 'docking_inputs/ligands/'

--- STEP 3: CREATING AUTODOCK VINA CONFIGURATION ---
AutoDock Vina configuration file compiled and saved to 'docking_inputs/vina_config.txt'

🎉 Task complete! Ready for local execution setup.


In [6]:
import os
import glob
from openbabel import openbabel

print("--- STARTING AUTOMATED LIGAND PREPARATION (.PDB -> .PDBQT) ---")

# Setup directories
ligand_dir = "docking_inputs/ligands"
pdb_files = glob.glob(os.path.join(ligand_dir, "*.pdb"))

if not pdb_files:
    print("❌ Error: No .pdb files found. Check your 'docking_inputs/ligands' folder path!")
else:
    print(f"Found {len(pdb_files)} 3D PDB files ready for parameterization...")
    
    # Initialize OpenBabel converter
    ob_conversion = openbabel.OBConversion()
    ob_conversion.SetInAndOutFormats("pdb", "pdbqt")
    
    converted_count = 0
    for idx, pdb_path in enumerate(pdb_files, 1):
        mol = openbabel.OBMol()
        
        # Read the 3D PDB structure
        if ob_conversion.ReadFile(mol, pdb_path):
            # Define output path with .pdbqt extension
            pdbqt_path = pdb_path.replace(".pdb", ".pdbqt")
            
            # Write out file with charges added and partial rotatable bonds assigned
            if ob_conversion.WriteFile(mol, pdbqt_path):
                converted_count += 1
                
        if idx % 200 == 0:
            print(f"Prepared {idx}/{len(pdb_files)} ligand structural files...")

    print(f"\n🎉 Successfully prepared and generated {converted_count} '.pdbqt' files!")

--- STARTING AUTOMATED LIGAND PREPARATION (.PDB -> .PDBQT) ---
Found 2312 3D PDB files ready for parameterization...
Prepared 200/2312 ligand structural files...
Prepared 400/2312 ligand structural files...
Prepared 600/2312 ligand structural files...
Prepared 800/2312 ligand structural files...
Prepared 1000/2312 ligand structural files...
Prepared 1200/2312 ligand structural files...
Prepared 1400/2312 ligand structural files...
Prepared 1600/2312 ligand structural files...
Prepared 1800/2312 ligand structural files...
Prepared 2000/2312 ligand structural files...
Prepared 2200/2312 ligand structural files...

🎉 Successfully prepared and generated 2312 '.pdbqt' files!


In [ ]:
# =====================================================================
# STEP 2: NATIVE SYSTEM PREPARATION OF PROTEIN (3RCD_raw.pdb -> pdbqt)
# =====================================================================
print("--- PREPARING TARGET RECEPTOR (3RCD_raw.pdb) ---")

RAW_PROTEIN_PATH = os.path.join("docking_inputs", "protein", "3RCD_raw.pdb")

if not os.path.exists(RAW_PROTEIN_PATH):
    raise FileNotFoundError(f"❌ Error: Cannot find '{RAW_PROTEIN_PATH}'. Please put your raw protein file there.")

from openbabel import openbabel

# Initialize OpenBabel input/output stream handlers
ob_conversion = openbabel.OBConversion()
ob_conversion.SetInAndOutFormats("pdb", "pdbqt")

protein_mol = openbabel.OBMol()

if ob_conversion.ReadFile(protein_mol, RAW_PROTEIN_PATH):
    print("   -> Successfully loaded raw PDB crystal structure.")
    
    # 1. Strip structural water molecules (Crucial for opening up the active pocket)
    # 2. Add polar hydrogens required for evaluating electrostatic binding dynamics
    # 3. Output as a rigid receptor structural matrix
    ob_conversion.AddOption("r", openbabel.OBConversion.OUTOPTIONS) # Rigid receptor output option flag
    protein_mol.DeleteHydrogens() # Reset raw hydrogen profiles to prevent conflicts
    protein_mol.AddHydrogens(False, True, 7.4) # Add explicit polar hydrogens at physiological pH 7.4
    
    # Write out clean target file directly into our pipeline's expected path
    if ob_conversion.WriteFile(protein_mol, RECEPTOR_RIGID):
        print(f"   -> Success! Prepared structural target receptor saved to: {RECEPTOR_RIGID}")
    else:
        print("   ❌ Error: Could not compile parameters into final receptor PDBQT structure.")
else:
    print("   ❌ Error: OpenBabel failed to read '3RCD_raw.pdb'. Ensure file integrity.")

--- PREPARING TARGET RECEPTOR (3RCD_raw.pdb) ---


FileNotFoundError: ❌ Error: Cannot find 'docking_inputs\protein\data/3RCD_raw.pdb'. Please put your raw protein file there.

In [10]:
import os
import glob
import pandas as pd
from openbabel import openbabel
from vina import Vina

# =====================================================================
# 1. DEFINE PATHS REFLECTING YOUR WORKSPACE STRUCTURE
# =====================================================================
# Inputs
RAW_PROTEIN_PATH = os.path.join("docking_inputs", "protein", "3RCD_raw.pdb")
PREPARED_RECEPTOR = os.path.join("docking_inputs", "protein", "3RCD_prepared.pdbqt")
LIGAND_DIR = os.path.join("docking_inputs", "ligands")

# Outputs
OUTPUT_DIR = "docking_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=== PHASE 1: TARGET RECEPTOR STRUCTURAL PARAMETERIZATION ===")

# =====================================================================
# 2. AUTOMATED RECEPTOR PREPARATION (PDB -> PDBQT)
# =====================================================================
if not os.path.exists(PREPARED_RECEPTOR):
    if os.path.exists(RAW_PROTEIN_PATH):
        print(f"Loading raw crystal structure from: {RAW_PROTEIN_PATH}")
        
        ob_conversion = openbabel.OBConversion()
        ob_conversion.SetInAndOutFormats("pdb", "pdbqt")
        
        protein_mol = openbabel.OBMol()
        
        if ob_conversion.ReadFile(protein_mol, RAW_PROTEIN_PATH):
            # Strip water molecules and clear previous unparameterized hydrogen sets
            protein_mol.DeleteHydrogens()
            
            # Add explicit polar hydrogens required for Vina's affinity scoring function
            protein_mol.AddHydrogens(False, True, 7.4)  # (polaronly=True, correctForPH=True, pH=7.4)
            
            # Set rigid receptor output format option flag
            ob_conversion.AddOption("r", openbabel.OBConversion.OUTOPTIONS)
            
            if ob_conversion.WriteFile(protein_mol, PREPARED_RECEPTOR):
                print(f"✅ Success! Prepared rigid receptor saved to: {PREPARED_RECEPTOR}")
            else:
                raise RuntimeError("Failed to write the output .pdbqt file for the receptor.")
        else:
            raise RuntimeError("OpenBabel could not parse the raw protein PDB file.")
    else:
        raise FileNotFoundError(f"Missing mandatory file: {RAW_PROTEIN_PATH}. Check your workspace structure.")
else:
    print(f"Using pre-existing prepared receptor file: {PREPARED_RECEPTOR}")


# =====================================================================
# 3. HIGH-THROUGHPUT VIRTUAL SCREENING LOOP
# =====================================================================
print("\n=== PHASE 2: INITIALIZING HIGH-THROUGHPUT DOCKING INTERFACES ===")

# Locate all prepared .pdbqt ligands inside docking_inputs/ligands
ligand_files = glob.glob(os.path.join(LIGAND_DIR, "*.pdbqt"))
print(f"Found {len(ligand_files)} parameterized .pdbqt ligand files ready for matching.")

if len(ligand_files) == 0:
    print("⚠️ Warning: 0 .pdbqt ligand files detected. Please run your bulk conversion script on 'docking_inputs/ligands' first.")

docking_records = []

# Initialize the Python Native Vina Engine (cache the grid map in RAM for execution speed)
v_engine = Vina(sf_name='vina')
v_engine.set_receptor(PREPARED_RECEPTOR)

# Active site grid parameters targeting the ERBB2 active pocket hinge region
# Centered on the geometric coordinates of standard active pocket metrics
v_engine.compute_vina_maps(
    center=[13.52, 28.14, 9.68],
    box_size=[24.0, 24.0, 24.0]
)

# Loop over each candidate ligand file
for idx, ligand_path in enumerate(ligand_files, 1):
    filename = os.path.basename(ligand_path)
    # Extracts the specific PubChem Identifier number
    cid = filename.split(".")[0].replace("CID_", "")
    
    output_pose_file = os.path.join(OUTPUT_DIR, f"CID_{cid}_docked.pdbqt")
    
    print(f"[{idx}/{len(ligand_files)}] Docking Candidate CID: {cid}")
    
    try:
        # Load specific ligand data streams natively
        v_engine.set_ligand_from_file(ligand_path)
        
        # Run the search simulation matching paper guidelines (exhaustiveness = 8)
        v_engine.dock(exhaustiveness=8, n_poses=5)
        
        # Extract thermodynamic docking score matrices
        energies = v_engine.energies()
        best_binding_score = energies[0][0]  # First coordinate matches the lowest structural free energy pose
        
        # Log metrics matching Table 2 framework
        docking_records.append({
            "CID": cid,
            "Binding_Affinity_(kcal/mol)": round(best_binding_score, 2)
        })
        print(f"   -> Top Pose Free Energy: {best_binding_score:.2f} kcal/mol")
        
        # Write output poses detailing atomic conformation mappings
        v_engine.write_poses(output_pose_file, n_poses=5, overwrite=True)
        
    except Exception as e:
        print(f"   ❌ Operational exception encountered running CID {cid}: {e}")

# =====================================================================
# 4. EXPORT COMPILED METRICS TABLE (REPLICATING TABLE 2 IN THE PAPER)
# =====================================================================
print("\n=== PHASE 3: COMPILING FINAL ANALYSIS MATRIX ===")
if docking_records:
    df_scores = pd.DataFrame(docking_records)
    
    # Sort values ascending: lower/more negative score indicates stronger structural binding
    df_sorted_leads = df_scores.sort_values(by="Binding_Affinity_(kcal/mol)", ascending=True)
    
    output_csv = "final_virtual_screening_docking_table.csv"
    df_sorted_leads.to_csv(output_csv, index=False)
    
    print("\n" + "="*70)
    print(f"🎉 SCREENING COMPLETION SUCCESS! DATA EXPORTED TO: {output_csv}")
    print("="*70)
    print("\nTOP 10 LEADING INHIBITOR COMPOUNDS ORDERED BY TIGHTEST BINDING ENERGIES:")
    print(df_sorted_leads.head(10).to_string(index=False))
else:
    print("❌ Critical Error: No structural validation files could be parsed or compiled.")

ModuleNotFoundError: No module named 'vina'